## Part 1 - Dataset & Preprocessing

In [ ]:
# ================================
# 1️. Import Libraries
# ================================
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

# ================================
# 2️. Load Dataset
# ================================
df = pd.read_csv(r"C:\Users\lukma\SAIA2163-NLP-Final-Project\Training_Data_Google_Play_reviews_6000.csv")

print("==== Dataset Preview ====")
print(df.head())

print("\n==== Dataset Info ====")
print(df.info())

print("\n==== Missing Values ====")
print(df.isnull().sum())

# ================================
# 3️. Strip whitespace and filter English reviews only
# ================================
df['userLang'] = df['userLang'].str.strip()
df_en = df[df['userLang'] == 'EN'].copy()
print("\n==== English Reviews Count ====")
print(len(df_en))

# ================================
# 4️. Map app_id to default app names
# ================================
app_mapping = {
    'org.telegram.messenger': 'Telegram',
    'com.facebook.orca': 'Facebook Messenger',
    'com.whatsapp': 'WhatsApp',
    'com.viber.voip': 'Viber',
    'com.snapchat.android': 'Snapchat',
    'com.tencent.mm': 'WeChat'
}

df_en['app_name'] = df_en['app_id'].map(app_mapping)
print("\n==== Sample App Names Mapping ====")
print(df_en[['app_id', 'app_name']].head(10))

# ================================
# 5️. Map scores to sentiment
# ================================
def map_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df_en['Sentiment'] = df_en['score'].apply(map_sentiment)
print("\n==== Sentiment Distribution ====")
print(df_en['Sentiment'].value_counts())

==== Dataset Preview ====
                               reviewId           userName  \
0  495266a4-f451-48c3-a844-fb3c07560d55     Foysal Hossain   
1  947fcd83-7a28-403d-b03b-d0bc20f52e0e          S K VERMA   
2  65856211-67ba-4560-84dd-a0055775ed90      Amanuel Abara   
3  cd5ba250-3a26-43b4-a378-77d18f73a503  Vagarangas X Aopi   
4  e8e886b4-d6c6-416b-b0a1-be90320c4024       Shafin islam   

                                           userImage  \
0  https://play-lh.googleusercontent.com/a-/ALV-U...   
1  https://play-lh.googleusercontent.com/a/ACg8oc...   
2  https://play-lh.googleusercontent.com/a/ACg8oc...   
3  https://play-lh.googleusercontent.com/a/ACg8oc...   
4  https://play-lh.googleusercontent.com/a-/ALV-U...   

                      content  score  thumbsUpCount reviewCreatedVersion  \
0  Gett van for no reason 😂😂😂      1              0                  NaN   
1       better' than WhatsApp      4              0                  NaN   
2    That was good app for me      5

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lukma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lukma\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lukma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\lukma\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# ================================
# 6️. Convert Review Dates to Datetime
# ================================
df_en['review_date'] = pd.to_datetime(df_en['at'], errors='coerce')
print("\n==== Sample Review Dates ====")
print(df_en[['at', 'review_date']].head(10))

# ================================
# 7️. Preprocessing Function with Stemming
# ================================
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

def preprocess_text_stem(text):
    """Preprocess text with stemming"""
    if pd.isnull(text):
        return ''
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove punctuation but keep emojis and numbers
    text = re.sub(r'[^\w\s\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF]', '', text)
    # Tokenization
    words = nltk.word_tokenize(text)
    # Remove stopwords and apply stemming
    words = [ps.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

# ================================
# 8️ Apply Preprocessing
# ================================
df_en['clean_text'] = df_en['content'].apply(preprocess_text_stem)
print("\n==== Sample Preprocessed Reviews ====")
print(df_en[['content', 'clean_text', 'Sentiment', 'app_name']].head(10))

# ================================
# 9️. Save Preprocessed CSV
# ================================
df_final = df_en[['clean_text', 'Sentiment', 'app_name', 'review_date']]\
    .rename(columns={'clean_text':'text', 'Sentiment':'label'})

df_final.to_csv("preprocessed_reviews.csv", index=False)
print("\n==== Preprocessed dataset saved as 'preprocessed_reviews.csv' ====")


==== Sample Review Dates ====
                    at         review_date
0  2023-09-19 15:05:31 2023-09-19 15:05:31
1  2023-09-19 14:59:30 2023-09-19 14:59:30
2  2023-09-19 14:55:06 2023-09-19 14:55:06
3  2023-09-19 14:50:18 2023-09-19 14:50:18
4  2023-09-19 14:48:20 2023-09-19 14:48:20
5  2023-09-19 14:46:54 2023-09-19 14:46:54
6  2023-09-19 14:43:18 2023-09-19 14:43:18
7  2023-09-19 14:42:49 2023-09-19 14:42:49
8  2023-09-19 14:37:57 2023-09-19 14:37:57
9  2023-09-19 14:37:46 2023-09-19 14:37:46

==== Sample Preprocessed Reviews ====
                              content                          clean_text  \
0          Gett van for no reason 😂😂😂                 gett van reason 😂😂😂   
1               better' than WhatsApp                     better whatsapp   
2            That was good app for me                            good app   
3                        Love the app                            love app   
4                              🕳️🕳️🕳️                                 🕳🕳